# Lesson 04 - Eszközhasználati tervezési minta

Ebben a leckében meg fogod tanulni az **Eszközhasználat** tervezési mintát AI ügynökök számára a Microsoft Agent Framework (Python) használatával. Témáink:

- Funkcióeszközök definiálása az `@tool` dekorátorral és típusosított paraméterekkel
- Eszközsémák biztosítása, hogy a modell értse, mit csinál minden eszköz
- Az eszközvégrehajtás vezérlése az `approval_mode` segítségével
- **Strukturált kimenet** visszaadása Pydantic modellek és a `response_format` segítségével

A szcenárió egy **utazási foglaló ügynök**, amely képes helyszíneket keresni, elérhetőséget ellenőrizni, és repülőjáratok információit lekérni.


## Beállítás


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Eszközök definiálása az @tool dekorátorral

Az `@tool` dekorátor egy egyszerű Python függvényt eszközzé alakít át, amelyet egy ügynök meghívhat.
Főbb pontok:

- A **docstring** lesz az eszköz leírása, amit a modell lát.
- A **típusannotációk** (beleértve az ismertetéseket tartalmazó `Annotated`-t is) határozzák meg az eszköz sémáját.
- Az `approval_mode` szabályozza, hogy a felhasználónak jóvá kell-e hagynia minden egyes hívást a végrehajtás előtt.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Több eszközzel rendelkező ügynök létrehozása

Adja át mindhárom eszközt az ügyfélnek, hogy a modell bármelyiket meghívhassa, amelyre szüksége van a felhasználó kérdésének megválaszolásához.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturált kimenet eszközökkel

Ha a `response_format` értékét egy Pydantic modellre állítjuk, az ágensek kénytelenek jól típusozott JSON objektumot visszaadni szabadszöveges helyett. Ez hasznos, ha a további kódnak programozottan kell feldolgoznia az eredményt.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Eszközjóváhagyási minták

Az `@tool` `approval_mode` paramétere szabályozza, hogy az eszközhívások emberi jóváhagyást igényelnek-e a végrehajtás előtt:

| Mód | Viselkedés |
|---|---|
| `"never_require"` | Az eszköz automatikusan fut — nincs szükség felhasználói megerősítésre. |
| `"always_require"` | Minden hívást a végrehajtás előtt jóvá kell hagynia a felhasználónak. |

Használd az `"always_require"` módot azoknál az eszközöknél, amelyek mellékhatásokkal járnak (pl. repülőjegy foglalás, hitelkártya terhelése), így az emberi beavatkozás biztosított marad.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Határozz meg eszközöket** az `@tool` dekorátor használatával, típusos paraméterekkel és docstringekkel, amelyek az eszköz sémájaként szolgálnak.
2. **Komponálj több eszközt**, hogy az ügynök sorozatban hívhassa meg őket összetett lekérdezések megválaszolásához.
3. **Adjon vissza strukturált kimenetet** úgy, hogy egy Pydantic modellt ad át `response_format`ként.
4. **Irányítsd az eszköz jóváhagyását** az `approval_mode` segítségével, hogy emberi közreműködő legyen jelen érzékeny műveletek esetén.

Ezek a minták képezik az alapját a megbízható, éles környezetbe szánt ügynökök felépítésének, amelyek biztonságosan képesek külső rendszerekkel kommunikálni.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Felelősségkizárás**:
Ezt a dokumentumot az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Kritikus információk esetén professzionális emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
